In [12]:
import re
from typing import Dict, List
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# Regex para PII estruturadas no padrão brasileiro
REGEX_PATTERNS = {
    "CPF": r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b|\b\d{11}b",
#    "EMAIL": r"\b[A-Za-Z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
    "EMAIL": r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+[a-zA-Z]{2,}\b",
    "TELEFONE": r"\b(?:\+55\s?)?(?:\(?\d{2}\)?\s?)?(?:9\d{4}|\d{4})[-\s]?\d{4}\b"
}

print("'Módulo de Regex para PII estruturadas carregado sem erros...")

print("Carregando o modelo de BERTimbau oficial NER...")

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first"
)

print("Modelo carregado com sucesso!.. ")

# Carrega as pipeline de Reconhecimento de Entidades Nomeadas usando o BERTimbau
#print("Carregando o modelo BERTimbau NER para portugues...")
#ner_pipeline = pipeline(
#    "ner",
#    model="Luciano/bertimbau-base-ner", # Modelo ajustado para entidades em PT-BR
#    aggregation_strategy="first"
#)
#print("Modelo de NLP carregado sem erros...")


# Função de Identificação e Anonimização
def anonimizar_texto(texto: str) -> Dict[str, object]:
    """
    Funcão identifica PII estruturadas usando (REGEX) e não estruturadas usando (BERTimbau NER)
    e substitui por tags de mascaramento.
    """
    texto_anonimizado = texto
    entidades_encontradas = []

    # Detecção por Regex (CPFs, E-mails, Telefones)
    for tipo_pii, pattern in REGEX_PATTERNS.items():
        matches = re.finditer(pattern, texto_anonimizado)
        for match in matches:
            valor = match.group()
            entidades_encontradas.append({"tipo": tipo_pii, "valor": valor, "metodo": "Regex"})
            texto_anonimizado = texto_anonimizado.replace(valor, f"[MASCARADO: {tipo_pii}]")

    # Detecção por NLP / NER (Nomes de Pessoas, Organizações, Locais)
    ner_results = ner_pipeline(texto)
    for ent in ner_results:
        valor = ent.get('word', '').replace('##', '')
        if len(valor) > 2 and valor not in texto_anonimizado:
            entidades_encontradas.append({"tipo": "ENTIDADE_NOME", "valor": valor, "metodo": "BERTimbau"})
        # Filtra entidades de Pessoas (PER) e Locais (LOC)
#        if ent['entity_group'] in ['PER', 'LOC']:
#            valor = ent['word']
#            tipo = "NOME_PESSOA" if ent['entity_group'] == 'PER' else "LOCALIZACAO"
#            entidades_encontradas.append({"tipo": tipo, "valor": valor, "metodo": "BERTimbau NER"})
#            texto_anonimizado = re.sub(rf"\b{re.escape(valor)}\b",f"[MASCARADO: {tipo}]", texto_anonimizado)

    return { 
        "texto_original": texto,
        "texto_anonimizado": texto_anonimizado,
        "pii_detectadas": entidades_encontradas
    }


# Texto teste
texto_exemplo = ( 
    "O voluntario Juninho Silva, portador do CPF 123.456.789-00 e e-mail juningameplays.silva@email.com, "
    "Solicitou a autorização para envio de dados do projeto ded apoio comunitario em Manaus."
)

resultado = anonimizar_texto(texto_exemplo)

print(">>>TEXTO ORIGINAL<<<")
print(resultado["texto_original"])
print("\n>>> TEXTO ANONIMIZADO <<<")
print(resultado["texto_anonimizado"])
print("\n>>> PII DETECTADAS <<<")
for pii in resultado["pii_detectadas"]:
    print(f"- [{pii['metodo']}] {pii['tipo']}: {pii['valor']}")

'Módulo de Regex para PII estruturadas carregado sem erros...
Carregando o modelo de BERTimbau oficial NER...


Loading weights: 100%|██████████████████████████████████████████████████| 197/197 [00:00<00:00, 28608.75it/s]
[transformers] BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias          

Modelo carregado com sucesso!.. 
>>>TEXTO ORIGINAL<<<
O voluntario Juninho Silva, portador do CPF 123.456.789-00 e e-mail juningameplays.silva@email.com, Solicitou a autorização para envio de dados do projeto ded apoio comunitario em Manaus.

>>> TEXTO ANONIMIZADO <<<
O voluntario Juninho Silva, portador do CPF [MASCARADO: CPF] e e-mail [MASCARADO: EMAIL], Solicitou a autorização para envio de dados do projeto ded apoio comunitario em Manaus.

>>> PII DETECTADAS <<<
- [Regex] CPF: 123.456.789-00
- [Regex] EMAIL: juningameplays.silva@email.com
- [BERTimbau] ENTIDADE_NOME: 123. 456. 789
- [BERTimbau] ENTIDADE_NOME: - 00
- [BERTimbau] ENTIDADE_NOME: juningameplays
- [BERTimbau] ENTIDADE_NOME: . silva @ email. com, Solicitou
